<p style="text-align: center">
<img src="../../assets/images/dtlogo.png" alt="Duckietown" width="50%">
</p>

# Training your Object Detector

In this notebook you will:
1. Install and set up [SAM3](https://github.com/facebookresearch/sam3) - a foundation model that automatically labels your images
2. Run SAM3 to generate YOLO-format bounding box labels for your collected dataset
3. Train a [YOLOv11n](https://docs.ultralytics.com/) object detection model on those labels
4. Inspect training results (PR curve, confusion matrix)
5. Export the trained model to ONNX for deployment on your Duckiebot

**Before starting:** make sure you have run the [Setup notebook](../02-Setup-Data-Collection/setup.ipynb) and that `assets/data/duckietown_dataset/` contains your `train/` and `val/` image splits.

## Hugging Face Authentication

### Create an account on Hugging Face

If you have not done so already, you need to [create an account on Hugging Face](https://huggingface.co/join).

### Request access to the SAM3 model

If you have not already you need to request access to the SAM3 model from facebook.

Check [the model page](https://huggingface.co/facebook/sam3) to request access and confirm you're approved. If you request access it can take a few minutes for your access to be approved.

### Set up your Hugging Face Access Token

Go to [the Hugging Face tokens settings](https://huggingface.co/settings/tokens) and click `Create new token`. In the settings for the token you can choose `Fine Grained` and make sure that the box labeled `Read access to contents of all public gated repos you can access` is checked. Once you have created your token, click `Copy` button to copy it to the clipboard. Open a terminal in the vscode and run:

```bash
hf auth login
```

Paste your token (from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)) when prompted, then come back here and run the next cell. You should see a `token is valid` printed on your screen. 

## Dataset

Your dataset lives in `assets/data/duckietown_dataset/` and was created by the Setup notebook. It should already contain two subdirectories:

```
duckietown_dataset/
  train/
    images/   ← your training images
    labels/   ← will be filled in by SAM3 auto-labeling below
  val/
    images/   ← your validation images
    labels/   ← will be filled in by SAM3 auto-labeling below
```

The label files (`.txt`) don't exist yet — SAM3 will create them automatically in the auto-labeling step.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import tempfile
import shutil
from typing import List
from datetime import datetime


def zip_sub_dirs(abs_root_dir: str, lst_rel_subdirs: List[str], output_basename: str) -> str:
    """Zip some sub-directories, return the zipped file's path"""
    out_full = f"{output_basename}.zip"
    if os.path.exists(out_full):
        print(f"File already exists at: {out_full}")
        print("Rename/Move it to run.\nNo operations performed.")
        return ""
    tmp_dir = tempfile.mkdtemp()
    print(f"[{datetime.now()}] Temporary directory created at: {tmp_dir}")
    original_paths = [os.path.join(abs_root_dir, _d) for _d in lst_rel_subdirs]
    tmp_paths = [os.path.join(tmp_dir, _d) for _d in lst_rel_subdirs]
    print(f"[{datetime.now()}] List of directories to include in the zip file:")
    for subdir in original_paths:
        assert os.path.exists(subdir), f"Specified path does not exist: {subdir}\nAbort!"
        print(f" - {subdir}")
    print(f"[{datetime.now()}] Move subdirs to the temp root dir")
    for ori, tmp in zip(original_paths, tmp_paths):
        shutil.move(ori, tmp)
    print(f"[{datetime.now()}] Compressing and creating the archive...")
    ret = shutil.make_archive(output_basename, 'zip', tmp_dir)
    print(f"[{datetime.now()}] Move subdirs back to original location")
    for tmp, ori in zip(tmp_paths, original_paths):
        shutil.move(tmp, ori)
    print(f"[{datetime.now()}] Finished. Archive created at: {ret}")
    return ret


# NOTE: DO NOT change these
ZIPPED_DATASET_BASENAME_FILE = "duckietown_dataset"
DATASET_DIR_ZIP = "../../assets/data/duckietown_dataset"
ZIPPED_DATASET_BASENAME_FULL = os.path.join(DATASET_DIR_ZIP, ZIPPED_DATASET_BASENAME_FILE)
TRAIN_DIR_ZIP = "train"
VALIDATION_DIR = "val"

_ = zip_sub_dirs(
    abs_root_dir=DATASET_DIR_ZIP,
    lst_rel_subdirs=[TRAIN_DIR_ZIP, VALIDATION_DIR],
    output_basename=ZIPPED_DATASET_BASENAME_FULL,
)


If everything went well, you should see the following output:

```
Finished. Archive created at: ../../assets/data/duckietown_dataset/duckietown_dataset.zip
```

> **Note:** This zip is optional for local training - it is only needed if you want to back up or share your dataset. You can skip this cell and proceed directly to the SAM3 setup below.

## Environment Setup

The cells below install [SAM3](https://github.com/facebookresearch/sam3) (Segment Anything Model 3) directly from the local clone in `assets/sam3/`. SAM3 is a vision-language foundation model from Meta that can locate and segment objects in images given a text prompt - we use it to automatically generate bounding box annotations for your dataset without any manual annotation.
This may take a couple of minutes the first time. Subsequent runs will skip the install if SAM3 is already present.

In [ ]:
import sys, re
from pathlib import Path

# Install dependencies
SAM3_DIR = Path("../../assets/sam3")
if not SAM3_DIR.exists():
    !git clone -q https://github.com/facebookresearch/sam3.git {SAM3_DIR}

pyproject = SAM3_DIR / "pyproject.toml"
pyproject.write_text(re.sub(r'"numpy==[^"]+",?\s*', "", pyproject.read_text(encoding="utf-8")), encoding="utf-8")

for py_file in SAM3_DIR.rglob("*.py"):
    text = py_file.read_text(encoding="utf-8")
    if "from __future__ import annotations" not in text:
        py_file.write_text("from __future__ import annotations\n" + text, encoding="utf-8")

!{sys.executable} -m pip install -q "{SAM3_DIR}"

sys.modules.pop("sam3", None)
if str(SAM3_DIR.resolve()) not in sys.path:
    sys.path.insert(0, str(SAM3_DIR.resolve()))


### GPU Check

SAM3 inference is significantly faster on a GPU - expect **~2–5s per image on CPU** vs **~0.1–0.5s on GPU**. For a dataset of 1000 images that's the difference between ~1h and ~5min. Training is also much faster with a GPU.

If no GPU is detected, training will still work but will be slow.

In [ ]:
# Check GPU
import torch
print(f"PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: No GPU found — training will be slow on CPU.")


## Dataset Paths & Imports

These paths point to the dataset created by the Setup notebook. If the assertions below fail, go back and run the [Setup notebook](../02-Setup-Data-Collection/setup.ipynb) first.

In [ ]:
# Paths
import os, yaml
import numpy as np
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
from ultralytics import YOLO

ASSETS_DIR   = (Path(os.getcwd()) / "../../assets").resolve()
DATASET_DIR  = ASSETS_DIR / "data" / "duckietown_dataset"
TRAIN_DIR    = DATASET_DIR / "train"
VAL_DIR      = DATASET_DIR / "val"
CLASSES_YAML = DATASET_DIR / "classes.yaml"

assert TRAIN_DIR.exists(), f"Train dir not found: {TRAIN_DIR}\nRun the Setup notebook first."
assert VAL_DIR.exists(),   f"Val dir not found: {VAL_DIR}\nRun the Setup notebook first."

print(f"Train images : {len(list((TRAIN_DIR / 'images').glob('*')))}")
print(f"Val images   : {len(list((VAL_DIR / 'images').glob('*')))}")


## Class Configuration

This writes a `classes.yaml` file that tells YOLO where your data lives and what classes to detect. The `names` dictionary maps integer class IDs to human-readable labels.

Add more entries if you want to detect additional objects, for example:

```yaml
names:
  0: 'yellow rubber duck'
  1: 'cone'
  2: 'duckiebot'
```

Make sure your SAM3 prompts below match the names you define here.

In [ ]:
# Write YOLO config
CLASSES_YAML.write_text(f"""train: {TRAIN_DIR}
val:   {VAL_DIR}

names:
  0: 'yellow rubber duck'  # add more classes here if needed
""")
print(CLASSES_YAML.read_text())


## Auto-Labeling with SAM3

SAM3 will iterate over every image in your `train/` and `val/` splits and generate a `.txt` label file for each one. Each line in a label file corresponds to one detected object and follows the YOLO format:

```
class_id  cx  cy  width  height
```

where all values are normalised to `[0, 1]` relative to the image dimensions.

**Confidence threshold:** detections below `confidence_threshold=0.3` are discarded. Raise this value if SAM3 is producing too many false positives; lower it if it is missing objects.

> **Tip:** If the run is interrupted you can re-run this cell safely - it will skip images that already have a label file.

In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor


def load_classes(classes_yaml):
    with open(classes_yaml) as f:
        cfg = yaml.safe_load(f)
    return [name for _, name in sorted(cfg["names"].items(), key=lambda kv: int(kv[0]))]


def xyxy_to_yolo_line(bbox, w, h, class_id):
    x1, y1, x2, y2 = bbox
    cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
    bw, bh = (x2 - x1) / w,     (y2 - y1) / h
    return f"{class_id} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"


class Sam3AutoLabel:
    def __init__(self, train_dir, val_dir, classes_yaml, confidence_threshold=0.3, device=None):
        self.train_dir = train_dir
        self.val_dir   = val_dir
        self.classes   = load_classes(classes_yaml)
        self.device    = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"SAM3 device: {self.device}")

        bpe = SAM3_DIR / "sam3" / "assets" / "bpe_simple_vocab_16e6.txt.gz"
        self.model = build_sam3_image_model(bpe_path=str(bpe))
        self.model.to(self.device).eval()
        self.processor = Sam3Processor(self.model,
                                        confidence_threshold=confidence_threshold,
                                        device=self.device)

    def _detect(self, img_path):
        image = Image.open(img_path).convert("RGB")
        w, h  = image.size
        ctx   = (torch.autocast("cuda", dtype=torch.bfloat16)
                 if self.device == "cuda"
                 else __import__("contextlib").nullcontext())
        dets  = []
        with torch.no_grad(), ctx:
            st = self.processor.set_image(image)
            for class_id, prompt in enumerate(self.classes):
                out    = self.processor.set_text_prompt(state=st, prompt=prompt)
                boxes  = out.get("boxes")
                scores = out.get("scores")
                if boxes is None:
                    continue
                for b, s in zip(boxes, scores):
                    s = float(s)
                    if s >= self.processor.confidence_threshold:
                        dets.append({"bbox": b.tolist(), "score": s, "class_id": class_id})
        return dets, (w, h)

    def _label_split(self, split):
        split_dir = self.train_dir if split == "train" else self.val_dir
        img_dir   = split_dir / "images"
        lbl_dir   = split_dir / "labels"
        imgs = [p for ext in ("*.jpg", "*.jpeg", "*.png") for p in img_dir.rglob(ext)]
        for img_path in tqdm(imgs, desc=f"Labeling {split}"):
            dets, (w, h) = self._detect(img_path)
            lines = [xyxy_to_yolo_line(d["bbox"], w, h, d["class_id"]) for d in dets]
            lbl   = lbl_dir / f"{img_path.stem}.txt"
            if lines:
                lbl.write_text("\n".join(lines))
            elif lbl.exists():
                lbl.unlink()

    def run(self):
        self._label_split("train")
        self._label_split("val")


autolabel = Sam3AutoLabel(
    train_dir=TRAIN_DIR,
    val_dir=VAL_DIR,
    classes_yaml=CLASSES_YAML,
    confidence_threshold=0.3,
)
autolabel.run()

## Verify Labels

Before training, it is worth visually checking that SAM3 produced sensible labels. The cell below picks the first labelled image from your training set and draws the predicted bounding boxes on it.

Check that:
- The boxes are tightly around the objects of interest
- There are no obvious false positives (boxes on background)
- The class labels are correct

If the labels look poor, try adjusting `confidence_threshold` in the auto-labeling cell above and re-run.

In [ ]:
train_labels = TRAIN_DIR / "labels"
train_images = TRAIN_DIR / "images"
label_files  = list(train_labels.glob("*.txt"))
assert label_files, f"No label files in {train_labels}"

label = label_files[0]
img   = next(
    (train_images / f"{label.stem}{ext}" for ext in (".png", ".jpg", ".jpeg")
     if (train_images / f"{label.stem}{ext}").exists()),
    None,
)
assert img, f"Image not found for label {label.name}"

image  = Image.open(img).convert("RGB")
w, h   = image.size
fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(image)
ax.axis("off")
classes = load_classes(CLASSES_YAML)
for line in label.read_text().splitlines():
    parts = line.split()
    if len(parts) != 5:
        continue
    cid, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:])
    x1, y1 = (cx - bw / 2) * w, (cy - bh / 2) * h
    ax.add_patch(plt.Rectangle((x1, y1), bw * w, bh * h, fill=False, linewidth=2))
    ax.text(x1, y1 - 2, classes[cid] if cid < len(classes) else str(cid),
            fontsize=10, bbox=dict(facecolor="black", alpha=0.5), color="white")
plt.show()


## Training

We train a **YOLOv11 nano** model (`yolo11n.yaml`) - the smallest and fastest variant, well suited for deployment on a Duckiebot's hardware.

Key parameters:

| Parameter | Value | Notes |
|-----------|-------|-------|
| `epochs` | 10 | Max training epochs - increase for better accuracy |
| `imgsz` | (480, 640) | Must match your camera resolution |
| `batch` | 16 | Reduce to 8 if you run out of GPU memory |
| `workers` | 0 | Set to 0 to avoid shared memory errors in Docker |

Training results (weights, plots, metrics) are saved to `assets/data/duckietown_dataset/runs/`.

In [ ]:
# Train - adjust epochs and batch as needed
model = YOLO("yolo11n.yaml")
model.train(
    data=str(CLASSES_YAML),
    epochs=10,
    imgsz=(480, 640),
    batch=16,
    workers=0,
    device=0 if torch.cuda.is_available() else "cpu",
    project=str(DATASET_DIR / "runs"),
    name="duckietown_detection",
    rect=True, # no padding waste on non-square 480x640
)

## Export to ONNX

[ONNX](https://onnx.ai/) (Open Neural Network Exchange) is a portable model format that can run on many different runtimes without needing PyTorch. We export to ONNX so the model can run efficiently on the Duckiebot using `onnxruntime`.

The exported model will be saved to `assets/best.onnx` - this is the file the Integration notebook loads.

Export settings:
- `opset=15` - ONNX opset version
- `simplify=True` - runs ONNX simplifier to reduce model size
- `nms=True` - bakes Non-Maximum Suppression into the graph so the output is already filtered otherwise you can implement NMS yourself and use it as a post processing step
- `half=True` - FP16 weights if GPU is available (smaller, faster)
- Output shape: `[1, 300, 6]` → up to 300 detections, each with `(x1, y1, x2, y2, score, class_id)`

In [ ]:
# Export to ONNX - saved directly to assets/best.onnx
runs     = sorted((DATASET_DIR / "runs").iterdir(), key=lambda p: p.stat().st_mtime)
last_run = runs[-1]
best_pt  = last_run / "weights" / "best.pt"

if not best_pt.exists():
    raise FileNotFoundError(f"best.pt not found at {best_pt}")

use_gpu = torch.cuda.is_available()
model   = YOLO(str(best_pt))
model.export(
    format="onnx",
    opset=15,
    imgsz=(480, 640),
    simplify=True,
    dynamic=False,
    nms=True,
    half=use_gpu,
    batch=1,
    device=0 if use_gpu else "cpu",
)

onnx_src = last_run / "weights" / "best.onnx"
onnx_dst = ASSETS_DIR / "best.onnx"
onnx_dst.write_bytes(onnx_src.read_bytes())
print(f"Model saved to: {onnx_dst}")
print("Done! Proceed to the next step.")


# Debugging and Model Inspection

Once you have finished training, there are a bunch of interesting outputs that will get generated during the training process that can be helpful for you to look at.

* After training, a `runs` directory has been created under `assets/data/duckietown_dataset/`
* Navigate to the folder in the VSCode file explorer on the left.
* After locating the `assets/data/duckietown_dataset/runs/` folder, go into the appropriate run directory inside.
* Then navigate to `runs/duckietown_detectionX/` where `X` is incremented each time you train.

In here you can see things like your PR curve, e.g.:

<img src="../../assets/images/PR_curve.png" alt="PR Curve" width="50%">

Your confusion matrix:

<img src="../../assets/images/confusion_matrix.png" alt="PR Curve" width="50%">

And sample training outputs:

<img src="../../assets/images/train_batch1.jpg" alt="PR Curve" width="50%">

# Next step

Onto the [Integration notebook](../04-Integration/integration.ipynb)!